# Exp 14: Attention dropping

Card: [exp14_attention_dropping.md](../../docs/neuroslope/experiments/exp14_attention_dropping.md)

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from PIL import Image

from core.boxes import AttentionDrop, read_boxes
from training.config import CLASSES, IMG_SIZE, ROOT, SEED, TRAIN_DIR
from training.config import VIP
from training.cv import CVResult
from training.runs import read_config
from dataset.dataset import read_train
from analysis.gradcam import GradCamFigures
from training.tracking import start_run
from analysis.plots import draw_boxes, preview, save_figures, use_style
from analysis.thresholds import best_f1

N_SHOW = 4
N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp14_attention_dropping" / "resnet18.yaml"
# trained in exp11 on folds 0-4, only read; compared on same folds as CONFIG
EXP11 = CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.yaml"
# case images of docs/neuroslope/methods/gradcam_augmentations.md, VIP cases
CASES = {
    "beside cat": 779,
    "on cat": 9101,
    "on rug": 2470,
}
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp14_attention_dropping"

# dataviz reference palette, light mode; default keeps exp06 baseline color
MUTED = "#898781"
SERIES = {"default (exp11)": "#eb6834", "attention_drop": "#2a78d6"}
use_style()

## attention_drop draws

In [ ]:
N_DRAWS = 6
train_df = read_train()
only_cat = train_df[CLASSES].eq([1, 0, 0, 0, 0]).all(axis=1)
picks = {cls: train_df[train_df[cls] == 1] for cls in VIP} | {
    "Cat only": train_df[only_cat]
}
image_ids = {
    name: rows.sample(1, random_state=SEED)["image_id"].item()
    for name, rows in picks.items()
}
images = {
    name: np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg")
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )
    for name, image_id in image_ids.items()
}
boxes = read_boxes()
BOX = SERIES["attention_drop"]
tfms = AttentionDrop(IMG_SIZE).compose()
show = preview(tfms)  # without Normalize, ToTensorV2


def draw(name, original=False):
    """(image, boxes, box labels): original, or one attention_drop draw. Boxes follow every transform."""
    image_boxes = boxes.get(image_ids[name], [])
    bboxes = np.array([b for _, b in image_boxes]).reshape(-1, 4)
    labels = [c for c, _ in image_boxes]
    if original:
        return images[name], bboxes, labels
    out = show(image=images[name], bboxes=bboxes, box_labels=labels)
    return out["image"], out["bboxes"], out["box_labels"]


fig, axes = plt.subplots(
    len(images), N_DRAWS + 1, figsize=(2.1 * (N_DRAWS + 1), 2.4 * len(images))
)
for name, row_axes in zip(images, axes):
    for i, ax in enumerate(row_axes):
        shown, bboxes, labels = draw(name, original=i == 0)
        ax.imshow(shown)
        for box, cls in zip(bboxes, labels):
            # dashed: area > 0.3 of image, never erased
            big = (box[2] - box[0]) * (box[3] - box[1]) > tfms.transforms[0].max_area
            draw_boxes(ax, shown, [box], [cls], BOX, ls="--" if big else "-")
        ax.set_axis_off()
        if name == VIP[0]:
            ax.set_title("Original" if i == 0 else f"Draw {i}", fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        name,
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {f"{PREFIX}_draws.png": fig}
plt.show()

## Training

In [ ]:
FOLDS = CVResult.load(CONFIG).folds
# exp11 result holds folds 0-4: keep folds of this experiment
results = {"default (exp11)": CVResult.load(EXP11).only(FOLDS)}

In [ ]:
results["attention_drop"] = CVResult.load(CONFIG)

## Training curves

In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, len(r.mean_curve("val_map")) + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(r.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="augmentations", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_training.png"] = fig
plt.show()

## Scores

In [ ]:
scores = pd.concat(
    {name: r.fold_scores() for name, r in results.items()}, names=["augmentations"]
)
mean = scores.groupby("augmentations", sort=False).mean()
pd.concat([scores, mean.set_index(mean.index + " mean")]).round(3)

In [ ]:
# difference to exp11 counts only if it has same sign on both folds
diff = (
    (scores.loc["attention_drop"] - scores.loc["default (exp11)"])
    .T.add_prefix("diff fold ")
)
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM: case images

In [ ]:
cams = {name: GradCamFigures(path) for name, path in {"default (exp11)": EXP11, "attention_drop": CONFIG}.items()}
cams["default (exp11)"].thresholds = best_f1(results["default (exp11)"].labels, results["default (exp11)"].probs, CLASSES)["threshold"]  # tuned on FOLDS only
case_classes = [
    (case, idx, cls)
    for case, idx in CASES.items()
    for cls in VIP
    if train_df.loc[idx, cls] == 1
]
columns = [(name, fold) for name in cams for fold in FOLDS]
fig, axes = plt.subplots(
    len(case_classes),
    len(columns) + 1,
    figsize=(2.6 * (len(columns) + 1), 3 * len(case_classes)),
    squeeze=False,
)
for (case, idx, cls), row_axes in zip(case_classes, axes):
    for k, (name, fold) in enumerate(columns):
        shown, cam, prob = cams[name].image_and_cam(idx, cls, fold)
        if k == 0:
            row_axes[0].imshow(shown)
            row_axes[0].set_title(f"{case} · {cls}", fontsize=9)
        row_axes[k + 1].imshow(shown)
        row_axes[k + 1].imshow(cam, cmap="jet", vmin=0, vmax=1, alpha=0.5)
        role = "held out" if train_df.loc[idx, "fold"] == fold else "trained"
        row_axes[k + 1].set_title(
            f"{name}\nfold {fold} ({role}) · p {prob:.3f}",
            fontsize=8,
            color=SERIES[name],
        )
for ax in axes.flat:
    ax.set_axis_off()
fig.tight_layout()
figures[f"{PREFIX}_grad_cam_cases.png"] = fig
plt.show()

## Grad-CAM: correct and incorrect (exp02)

In [ ]:
new_cams = cams["attention_drop"]
for (cls, outcome), fig in new_cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

## Grad-CAM: validation and train (exp12)

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = new_cams.positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "fast"}):
    mlflow.log_table(scores.reset_index(), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": new_cams.thresholds[c] for c in VIP})

save_figures(figures, FIGURES)